# Sci-epo render session · 渲染会话

1. **Runtime → Change runtime type** · **代码执行程序 → 更改运行时类型**: pick the runtime the session page names (G4 for H3 LITE). A rehearsal needs no GPU, so a CPU runtime is enough. 选择会话页写明的运行时（H3 LITE 用 G4）；彩排不用 GPU，选 CPU 即可。
2. **Run the cell below** · **运行下面的单元格**: click ▶.
3. **Paste the session code** · **粘贴会话码**: copy it from the session on Sci-epo's 制作 tab, paste it into the hidden prompt and press Enter. The code is not saved in this notebook. 从 Sci-epo 制作页的渲染会话复制会话码，粘贴到隐藏输入框后回车；会话码不会存进笔记本。
4. **Keep this tab open** · **保持此页打开**: progress shows here and on Sci-epo. When the session ends, the cell releases the runtime by itself, also after an error or a stop, unless you tick `KEEP_RUNTIME`. 进度同时显示在这里和 Sci-epo 上。会话结束后（出错或停止也一样），单元格会自动释放运行时；勾选 `KEEP_RUNTIME` 则保留。

The cell downloads the worker program from your Sci-epo server and runs it only if its SHA-256 matches what the server states. 单元格从你的 Sci-epo 服务器下载工作机程序，SHA-256 与服务器给出的一致才运行。

In [ ]:
#@title Sci-epo render session · 渲染会话
KEEP_RUNTIME = False  #@param {type:"boolean"}
import getpass, hashlib, json, os, re, subprocess, sys, urllib.error, urllib.parse, urllib.request
from google.colab import runtime

class NoRedirect(urllib.request.HTTPRedirectHandler):
    def redirect_request(self, req, fp, code, msg, headers, newurl):  # the token must not follow a redirect
        raise urllib.error.HTTPError(req.full_url, code, 'redirect refused', headers, fp)

def fetch(url, token):
    request = urllib.request.Request(url, headers={'authorization': f'Bearer {token}'})
    with urllib.request.build_opener(NoRedirect).open(request, timeout=60) as response:
        return response.read()

process = None
try:
    code = getpass.getpass('Session code 会话码: ').strip()
    server, _, token = code.partition('#')
    server = server.rstrip('/')
    host = urllib.parse.urlsplit(server)
    if not re.fullmatch(r'sciepo_ss_[A-Za-z0-9_-]{43}', token) or not (host.scheme == 'https' or host.hostname in ('127.0.0.1', 'localhost')):
        raise SystemExit('This is not a Sci-epo session code (https://…#sciepo_ss_…). 这不是 Sci-epo 的会话码。')
    try:
        session = json.loads(fetch(server + '/api/worker/v1/session', token))
        program = fetch(server + '/api/worker/v1/session/worker.py', token)
    except urllib.error.HTTPError as error:
        raise SystemExit(f'Sci-epo answered HTTP {error.code}: the code is wrong or expired, or the session has ended. 会话码无效、已过期或会话已结束。') from None
    if hashlib.sha256(program).hexdigest() != session['worker']['sha256']:
        raise SystemExit('The worker program does not match the server; it was not run. 工作机程序与服务器给出的校验值不符，没有运行。')
    with open('/content/sciepo_worker.py', 'wb') as handle:
        handle.write(program)
    process = subprocess.Popen([sys.executable, '/content/sciepo_worker.py', 'session'],
                               env=dict(os.environ, SCIEPO_SESSION=code, PYTHONUNBUFFERED='1'),
                               stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, encoding='utf-8', errors='replace')
    for line in process.stdout:
        print(line, end='')
    print(f'The worker ended with exit code {process.wait()}. 工作机已结束。')
finally:
    code = token = None
    try:
        if process and process.poll() is None:
            process.terminate()  # the worker ends the session as stopped, then exits
            try:
                process.wait(timeout=60)
            except subprocess.TimeoutExpired:
                process.kill()
    finally:
        if not KEEP_RUNTIME:
            runtime.unassign()